# ASTRAM AI - Comprehensive Exploratory Data Analysis

**Flipkart Grid 2.0 - Round 2: Event-Driven Congestion Management**

**Author**: SHIVAPREETHAM ROHITH  
**Date**: June 2026  
**Dataset**: 8,173 traffic incidents from Bengaluru (Nov 2023 - Apr 2024)

---

## Table of Contents

1. [Dataset Overview](#1.-Dataset-Overview)
2. [Temporal Analysis](#2.-Temporal-Analysis)
3. [Spatial Analysis](#3.-Spatial-Analysis)
4. [Cause Analysis](#4.-Cause-Analysis)
5. [Impact Analysis](#5.-Impact-Analysis)
6. [Corridor Intelligence](#6.-Corridor-Intelligence)
7. [Predictive Insights](#7.-Predictive-Insights)
8. [Key Findings & Recommendations](#8.-Key-Findings-&-Recommendations)

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully")

In [ ]:
# Load enhanced dataset
df = pd.read_parquet('../../astram/data/model_ready_v2.parquet')

print(f"Dataset loaded: {len(df):,} records with {len(df.columns)} columns")
print(f"Date range: {df['start_datetime'].min()} to {df['start_datetime'].max()}")
print(f"\nFirst 5 records:")
df.head()

---

## 1. Dataset Overview

### 1.1 Basic Statistics

In [ ]:
# Dataset summary
summary = {
    'Total Incidents': len(df),
    'Date Range': f"{df['start_datetime'].min().date()} to {df['start_datetime'].max().date()}",
    'Duration (days)': (df['start_datetime'].max() - df['start_datetime'].min()).days,
    'Unique Corridors': df['corridor'].nunique(),
    'Unique Police Stations': df['police_station'].nunique(),
    'Event Causes': df['event_cause'].nunique(),
    'Vehicle Types': df['veh_type'].nunique(),
    'Planned Events': (df['event_type'] == 'planned').sum(),
    'Unplanned Events': (df['event_type'] == 'unplanned').sum(),
    'Road Closures': df['requires_road_closure'].sum(),
    'Median Resolution Time (hours)': df['resolution_time_hours'].median(),
    'Kannada Descriptions': sum(1 for d in df['description'] if not pd.isna(d) and not str(d).isascii())
}

summary_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
print("\n" + "="*60)
print("DATASET OVERVIEW")
print("="*60)
print(summary_df.to_string(index=False))
print("="*60)

In [ ]:
# Missing values analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing Count': missing.values,
    'Missing %': missing_pct.values
}).sort_values('Missing Count', ascending=False).head(10)

print("\nTop 10 Columns with Missing Values:")
print(missing_df[missing_df['Missing Count'] > 0].to_string(index=False))

In [ ]:
# Geographic coverage
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot of incidents
ax1.scatter(df['longitude'], df['latitude'], alpha=0.3, s=5, c='red')
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')
ax1.set_title('Geographic Distribution of Incidents in Bengaluru')
ax1.grid(True, alpha=0.3)

# Hexbin density
hexbin = ax2.hexbin(df['longitude'], df['latitude'], gridsize=30, cmap='YlOrRd', mincnt=1)
ax2.set_xlabel('Longitude')
ax2.set_ylabel('Latitude')
ax2.set_title('Incident Density Heatmap')
plt.colorbar(hexbin, ax=ax2, label='Incident Count')

plt.tight_layout()
plt.savefig('../../project/docs/eda_geographic_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Geographic bounds: Lat [{df['latitude'].min():.4f}, {df['latitude'].max():.4f}], Lon [{df['longitude'].min():.4f}, {df['longitude'].max():.4f}]")

---

## 2. Temporal Analysis

### 2.1 Incidents Over Time

In [ ]:
# Daily incident trend
df['date'] = df['start_datetime'].dt.date
daily_counts = df.groupby('date').size().reset_index(name='count')
daily_counts['date'] = pd.to_datetime(daily_counts['date'])

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(daily_counts['date'], daily_counts['count'], linewidth=1.5, color='steelblue')
ax.fill_between(daily_counts['date'], daily_counts['count'], alpha=0.3, color='steelblue')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Incidents')
ax.set_title('Daily Incident Count Over Time', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../../project/docs/eda_daily_trend.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Average incidents per day: {daily_counts['count'].mean():.1f}")
print(f"Peak day: {daily_counts.loc[daily_counts['count'].idxmax(), 'date'].date()} with {daily_counts['count'].max()} incidents")

In [ ]:
# Hourly distribution
hourly = df['hour'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(hourly.index, hourly.values, color='coral', edgecolor='darkred', linewidth=1.5)

# Highlight rush hours
for i, bar in enumerate(bars):
    if 7 <= i <= 10 or 16 <= i <= 20:
        bar.set_color('crimson')
        bar.set_alpha(0.8)

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Number of Incidents')
ax.set_title('Incident Distribution by Hour (Rush Hours Highlighted in Red)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(range(0, 24))
plt.tight_layout()
plt.savefig('../../project/docs/eda_hourly_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

rush_morning = df[(df['hour'] >= 7) & (df['hour'] <= 10)]
rush_evening = df[(df['hour'] >= 16) & (df['hour'] <= 20)]
print(f"\nMorning rush (7-10 AM): {len(rush_morning):,} incidents ({len(rush_morning)/len(df)*100:.1f}%)")
print(f"Evening rush (4-8 PM): {len(rush_evening):,} incidents ({len(rush_evening)/len(df)*100:.1f}%)")
print(f"Peak hour: {hourly.idxmax()}:00 with {hourly.max():,} incidents")

In [ ]:
# Weekday vs Weekend analysis
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_counts = df['weekday'].value_counts().sort_index()
weekday_counts.index = [day_names[i] for i in weekday_counts.index]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Weekday bar chart
colors = ['steelblue']*5 + ['coral']*2
ax1.bar(range(7), weekday_counts.values, color=colors, edgecolor='black', linewidth=1.5)
ax1.set_xticks(range(7))
ax1.set_xticklabels(day_names, rotation=45, ha='right')
ax1.set_ylabel('Number of Incidents')
ax1.set_title('Incidents by Day of Week', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Weekend vs Weekday pie
weekend_count = df['is_weekend'].sum()
weekday_count = len(df) - weekend_count
ax2.pie([weekday_count, weekend_count], labels=['Weekday', 'Weekend'], 
        autopct='%1.1f%%', colors=['steelblue', 'coral'], startangle=90,
        textprops={'fontsize': 12, 'weight': 'bold'})
ax2.set_title('Weekday vs Weekend Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../../project/docs/eda_weekday_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nBusiest day: {weekday_counts.idxmax()} with {weekday_counts.max():,} incidents")
print(f"Quietest day: {weekday_counts.idxmin()} with {weekday_counts.min():,} incidents")

---

## 3. Spatial Analysis

### 3.1 Corridor Analysis

In [ ]:
# Top corridors by incident count
corridor_counts = df['corridor'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(range(len(corridor_counts)), corridor_counts.values, color='teal', edgecolor='darkslategray', linewidth=1.5)
ax.set_yticks(range(len(corridor_counts)))
ax.set_yticklabels(corridor_counts.index)
ax.set_xlabel('Number of Incidents')
ax.set_title('Top 15 Corridors by Incident Count', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (bar, value) in enumerate(zip(bars, corridor_counts.values)):
    ax.text(value + 10, i, f'{value:,}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../../project/docs/eda_top_corridors.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nTop 3 corridors account for {corridor_counts.head(3).sum():,} incidents ({corridor_counts.head(3).sum()/len(df)*100:.1f}%)")

In [ ]:
# Police station workload
station_counts = df['police_station'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(range(len(station_counts)), station_counts.values, color='darkviolet', edgecolor='purple', linewidth=1.5)
ax.set_yticks(range(len(station_counts)))
ax.set_yticklabels(station_counts.index)
ax.set_xlabel('Number of Incidents')
ax.set_title('Top 15 Police Stations by Incident Workload', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

for i, value in enumerate(station_counts.values):
    ax.text(value + 5, i, f'{value:,}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../../project/docs/eda_police_stations.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nMost burdened station: {station_counts.index[0]} with {station_counts.values[0]:,} incidents")

---

## 4. Cause Analysis

### 4.1 Event Causes Distribution

In [ ]:
# Event causes
cause_counts = df['event_cause'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Bar chart
ax1.bar(range(len(cause_counts)), cause_counts.values, color='indianred', edgecolor='darkred', linewidth=1.5)
ax1.set_xticks(range(len(cause_counts)))
ax1.set_xticklabels(cause_counts.index, rotation=45, ha='right')
ax1.set_ylabel('Number of Incidents')
ax1.set_title('Incident Count by Cause', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Pie chart (top 8 + others)
top_causes = cause_counts.head(8)
others = cause_counts.iloc[8:].sum()
pie_data = list(top_causes.values) + [others]
pie_labels = list(top_causes.index) + ['Others']

ax2.pie(pie_data, labels=pie_labels, autopct='%1.1f%%', startangle=90)
ax2.set_title('Cause Distribution (Top 8 + Others)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../../project/docs/eda_cause_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nDominant cause: {cause_counts.index[0]} ({cause_counts.values[0]:,} incidents, {cause_counts.values[0]/len(df)*100:.1f}%)")
print(f"Top 3 causes account for {cause_counts.head(3).sum()/len(df)*100:.1f}% of all incidents")

In [ ]:
# Cause vs Closure rate
cause_closure = df.groupby('event_cause').agg({
    'requires_road_closure': ['sum', 'count']
}).reset_index()
cause_closure.columns = ['Cause', 'Closures', 'Total']
cause_closure['Closure_Rate'] = (cause_closure['Closures'] / cause_closure['Total']) * 100
cause_closure = cause_closure.sort_values('Closure_Rate', ascending=False)

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(range(len(cause_closure)), cause_closure['Closure_Rate'], 
               color='orangered', edgecolor='darkred', linewidth=1.5)
ax.set_yticks(range(len(cause_closure)))
ax.set_yticklabels(cause_closure['Cause'])
ax.set_xlabel('Closure Rate (%)')
ax.set_title('Road Closure Rate by Incident Cause', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

for i, (rate, total) in enumerate(zip(cause_closure['Closure_Rate'], cause_closure['Total'])):
    ax.text(rate + 0.5, i, f'{rate:.1f}% (n={total})', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../../project/docs/eda_cause_closure_rate.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nHighest closure rate: {cause_closure.iloc[0]['Cause']} ({cause_closure.iloc[0]['Closure_Rate']:.1f}%)")

In [ ]:
# Vehicle type breakdown
veh_counts = df['veh_type'].value_counts()

fig, ax = plt.subplots(figsize=(12, 7))
wedges, texts, autotexts = ax.pie(veh_counts.values, labels=veh_counts.index, 
                                    autopct='%1.1f%%', startangle=90,
                                    textprops={'fontsize': 11})
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_weight('bold')

ax.set_title('Vehicle Type Distribution in Incidents', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../project/docs/eda_vehicle_types.png', dpi=300, bbox_inches='tight')
plt.show()

public_transit = df[df['veh_type'].isin(['BMTC Bus', 'KSRTC Bus'])]
print(f"\nPublic transit incidents (BMTC + KSRTC): {len(public_transit):,} ({len(public_transit)/len(df)*100:.1f}%)")

---

## 5. Impact Analysis

### 5.1 Resolution Time Analysis

In [ ]:
# Resolution time distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
ax1.hist(df['resolution_time_hours'], bins=50, color='skyblue', edgecolor='steelblue', linewidth=1.5)
ax1.axvline(df['resolution_time_hours'].median(), color='red', linestyle='--', linewidth=2, label=f'Median: {df["resolution_time_hours"].median():.2f}h')
ax1.set_xlabel('Resolution Time (hours)')
ax1.set_ylabel('Frequency')
ax1.set_title('Distribution of Incident Resolution Times', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Box plot by cause
top_causes_list = df['event_cause'].value_counts().head(10).index
df_top_causes = df[df['event_cause'].isin(top_causes_list)]
df_top_causes.boxplot(column='resolution_time_hours', by='event_cause', ax=ax2, rot=45)
ax2.set_xlabel('Cause')
ax2.set_ylabel('Resolution Time (hours)')
ax2.set_title('Resolution Time by Cause (Top 10)', fontsize=14, fontweight='bold')
plt.suptitle('')  # Remove default title

plt.tight_layout()
plt.savefig('../../project/docs/eda_resolution_time.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nResolution Time Statistics:")
print(f"  Median: {df['resolution_time_hours'].median():.2f} hours")
print(f"  Mean: {df['resolution_time_hours'].mean():.2f} hours")
print(f"  90th percentile: {df['resolution_time_hours'].quantile(0.9):.2f} hours")
print(f"  Max: {df['resolution_time_hours'].max():.2f} hours")

In [ ]:
# Resolution time by corridor
corridor_resolution = df.groupby('corridor')['resolution_time_hours'].agg(['median', 'count']).reset_index()
corridor_resolution = corridor_resolution[corridor_resolution['count'] >= 50].sort_values('median', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(range(len(corridor_resolution)), corridor_resolution['median'], 
               color='salmon', edgecolor='darkred', linewidth=1.5)
ax.set_yticks(range(len(corridor_resolution)))
ax.set_yticklabels(corridor_resolution['corridor'])
ax.set_xlabel('Median Resolution Time (hours)')
ax.set_title('Median Resolution Time by Corridor (Min 50 incidents)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

for i, (median, count) in enumerate(zip(corridor_resolution['median'], corridor_resolution['count'])):
    ax.text(median + 0.05, i, f'{median:.2f}h (n={count})', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../../project/docs/eda_corridor_resolution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Road closure analysis
closure_stats = {
    'Total Closures': df['requires_road_closure'].sum(),
    'Closure Rate': f"{(df['requires_road_closure'].sum()/len(df)*100):.1f}%",
    'Avg Resolution (Closure)': f"{df[df['requires_road_closure']==True]['resolution_time_hours'].mean():.2f}h",
    'Avg Resolution (No Closure)': f"{df[df['requires_road_closure']==False]['resolution_time_hours'].mean():.2f}h"
}

print("\nRoad Closure Impact:")
for key, value in closure_stats.items():
    print(f"  {key}: {value}")

# Closure vs No Closure comparison
fig, ax = plt.subplots(figsize=(10, 6))
closure_data = [df[df['requires_road_closure']==True]['resolution_time_hours'], 
                df[df['requires_road_closure']==False]['resolution_time_hours']]
ax.boxplot(closure_data, labels=['With Closure', 'No Closure'])
ax.set_ylabel('Resolution Time (hours)')
ax.set_title('Resolution Time: Road Closure vs No Closure', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../../project/docs/eda_closure_impact.png', dpi=300, bbox_inches='tight')
plt.show()

---

## 6. Corridor Intelligence

### 6.1 Corridor Stress Analysis

In [ ]:
# Corridor intelligence metrics
corridor_intel = df[df['corridor'] != 'Non-corridor'].groupby('corridor').agg({
    'id': 'count',
    'requires_road_closure': 'sum',
    'resolution_time_hours': 'median'
}).reset_index()
corridor_intel.columns = ['Corridor', 'Total_Incidents', 'Closures', 'Median_Resolution']
corridor_intel['Closure_Rate'] = (corridor_intel['Closures'] / corridor_intel['Total_Incidents']) * 100

# Calculate stress index (normalized composite metric)
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
corridor_intel['Freq_Norm'] = scaler.fit_transform(corridor_intel[['Total_Incidents']])
corridor_intel['Closure_Norm'] = scaler.fit_transform(corridor_intel[['Closure_Rate']])
corridor_intel['Resolution_Norm'] = scaler.fit_transform(corridor_intel[['Median_Resolution']])
corridor_intel['Stress_Index'] = (
    0.4 * corridor_intel['Freq_Norm'] +
    0.4 * corridor_intel['Closure_Norm'] +
    0.2 * corridor_intel['Resolution_Norm']
) * 100

corridor_intel = corridor_intel.sort_values('Stress_Index', ascending=False).head(15)

print("\nTop 15 Corridors by Stress Index:")
print(corridor_intel[['Corridor', 'Total_Incidents', 'Closure_Rate', 'Stress_Index']].to_string(index=False))

In [ ]:
# Stress index visualization
fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(range(len(corridor_intel)), corridor_intel['Stress_Index'], 
               color='crimson', edgecolor='darkred', linewidth=1.5)
ax.set_yticks(range(len(corridor_intel)))
ax.set_yticklabels(corridor_intel['Corridor'])
ax.set_xlabel('Stress Index (0-100)')
ax.set_title('Corridor Stress Index (Top 15)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

for i, (stress, total) in enumerate(zip(corridor_intel['Stress_Index'], corridor_intel['Total_Incidents'])):
    ax.text(stress + 1, i, f'{stress:.1f} (n={total})', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../../project/docs/eda_corridor_stress.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Corridor-Hour heatmap
top_corridors = df['corridor'].value_counts().head(10).index
corridor_hour = df[df['corridor'].isin(top_corridors)].groupby(['corridor', 'hour']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(corridor_hour, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Incident Count'})
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Corridor')
ax.set_title('Incident Heatmap: Top 10 Corridors × Hour of Day', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../project/docs/eda_corridor_hour_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nThis heatmap helps identify:")
print("  - Peak hours for each corridor")
print("  - Resource pre-positioning opportunities")
print("  - Predictable congestion patterns")

---

## 7. Predictive Insights

### 7.1 Feature Correlations

In [ ]:
# Correlation matrix for numeric features
numeric_cols = ['hour', 'weekday', 'month', 'is_weekend', 'is_night', 'is_holiday',
                'days_since_last_incident', 'moon_phase', 'resolution_time_hours',
                'requires_road_closure']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../project/docs/eda_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nKey Correlations with Resolution Time:")
res_corr = corr_matrix['resolution_time_hours'].sort_values(ascending=False)
print(res_corr[res_corr.index != 'resolution_time_hours'].head(5))

In [ ]:
# Planned vs Unplanned events comparison
planned_stats = df[df['event_type'] == 'planned'].agg({
    'id': 'count',
    'requires_road_closure': lambda x: (x.sum()/len(x)*100),
    'resolution_time_hours': 'median'
})

unplanned_stats = df[df['event_type'] == 'unplanned'].agg({
    'id': 'count',
    'requires_road_closure': lambda x: (x.sum()/len(x)*100),
    'resolution_time_hours': 'median'
})

comparison = pd.DataFrame({
    'Planned': planned_stats,
    'Unplanned': unplanned_stats
}, index=['Count', 'Closure Rate (%)', 'Median Resolution (h)'])

print("\nPlanned vs Unplanned Events:")
print(comparison)

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Count
axes[0].bar(['Planned', 'Unplanned'], [planned_stats['id'], unplanned_stats['id']], 
            color=['steelblue', 'coral'])
axes[0].set_ylabel('Count')
axes[0].set_title('Incident Count', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Closure rate
axes[1].bar(['Planned', 'Unplanned'], 
            [planned_stats['requires_road_closure'], unplanned_stats['requires_road_closure']], 
            color=['steelblue', 'coral'])
axes[1].set_ylabel('Closure Rate (%)')
axes[1].set_title('Road Closure Rate', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

# Resolution time
axes[2].bar(['Planned', 'Unplanned'], 
            [planned_stats['resolution_time_hours'], unplanned_stats['resolution_time_hours']], 
            color=['steelblue', 'coral'])
axes[2].set_ylabel('Median Resolution (hours)')
axes[2].set_title('Resolution Time', fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../../project/docs/eda_planned_vs_unplanned.png', dpi=300, bbox_inches='tight')
plt.show()

---

## 8. Key Findings & Recommendations

### 8.1 Summary Statistics

In [ ]:
print("="*80)
print("KEY FINDINGS - ASTRAM AI EDA")
print("="*80)

findings = [
    ("Dataset Coverage", f"{len(df):,} incidents over {(df['start_datetime'].max() - df['start_datetime'].min()).days} days"),
    ("Geographic Scope", f"{df['corridor'].nunique()} corridors, {df['police_station'].nunique()} police stations"),
    ("Dominant Cause", f"{df['event_cause'].value_counts().index[0]} ({df['event_cause'].value_counts().values[0]/len(df)*100:.1f}%)"),
    ("Peak Hour", f"{df['hour'].value_counts().index[0]}:00 ({df['hour'].value_counts().values[0]:,} incidents)"),
    ("Busiest Day", f"{day_names[df['weekday'].value_counts().index[0]]} ({df['weekday'].value_counts().values[0]:,} incidents)"),
    ("Road Closure Rate", f"{df['requires_road_closure'].sum()/len(df)*100:.1f}%"),
    ("Median Resolution", f"{df['resolution_time_hours'].median():.2f} hours"),
    ("Unplanned Events", f"{(df['event_type']=='unplanned').sum()/len(df)*100:.1f}%"),
    ("Top Corridor", f"{df['corridor'].value_counts().index[0]} ({df['corridor'].value_counts().values[0]:,} incidents)"),
    ("Public Transit Impact", f"{len(df[df['veh_type'].isin(['BMTC Bus', 'KSRTC Bus'])]):,} BMTC/KSRTC incidents")
]

for finding, value in findings:
    print(f"  {finding:.<50} {value}")

print("="*80)

In [ ]:
print("\n" + "="*80)
print("RECOMMENDATIONS FOR FORECASTING MODEL")
print("="*80)

recommendations = [
    "1. Focus on vehicle_breakdown prediction - accounts for 59.9% of incidents",
    "2. Model resolution_time_hours as regression target (median: 0.97h)",
    "3. Use corridor_tier as key feature - strong correlation with impact",
    "4. Incorporate hour and weekday - clear temporal patterns observed",
    "5. Add weather data for water_logging prediction (rain correlation expected)",
    "6. Pre-position resources on high-stress corridors (Mysore Road, Bellary Road)",
    "7. Separate models for planned vs unplanned events (different characteristics)",
    "8. Predict closure probability separately - 8.3% base rate",
    "9. Use historical days_since_last_incident for temporal trends",
    "10. Translate Kannada descriptions for NLP features (904 descriptions)"
]

for rec in recommendations:
    print(f"  {rec}")

print("="*80)

---

## Export Report to PDF

To export this notebook to PDF:

```bash
jupyter nbconvert --to pdf 02_eda_analysis.ipynb --output ../../project/docs/EDA_Report.pdf
```

Or use File → Download as → PDF in Jupyter interface.

In [ ]:
print("\n" + "="*80)
print("EDA ANALYSIS COMPLETE")
print("="*80)
print(f"\nGenerated {len([f for f in __import__('os').listdir('../../project/docs') if 'eda_' in f])} visualization files in project/docs/")
print("\nNext steps:")
print("  1. Export this notebook to PDF for judges")
print("  2. Use insights to build forecasting model")
print("  3. Integrate weather API for water_logging prediction")
print("  4. Create planned events database")
print("\nAuthor: SHIVAPREETHAM ROHITH")
print("Date: June 2026")
print("="*80)